In [0]:
from pyspark.sql import functions as f
import sys
sys.path.append('..')
sys.path.append('../..')

import lib_dna_member.generate_population as gp
import lib_dna_member.job_manager as job_manager
import lib_dna_member.transaction_features as features
from lib_dna_member.s3 import member_dna_input_data_validator
from databricks.feature_engineering import FeatureEngineeringClient

In [0]:
%run ../../config/utils

In [0]:
def generate_transaction(job):
    """
    Generate transaction based variables for the given population
    Parameters:
        job (object): Job Manager object based on the current config file

    Returns:
        dna (pyspark.sql.DataFrame): Transaction data of the given population
    """

    dna = job.tables["population"]
    orig_cols = dna.columns

    dna = features.feature_stdev(job, dna)

    dna = features.feature_trips(job, dna)

    dna = features.feature_spend(job, dna)

    dna = features.feature_spend_in_store(job, dna)

    dna = features.feature_units(job, dna)

    dna = features.feature_units_over_fifty(job, dna)

    dna = features.feature_gas_trips(job, dna)

    dna = features.feature_gas_distinct_days(job, dna)

    dna = features.feature_gas_spend(job, dna)

    dna = features.feature_gas_and_store_distinct_days(job, dna)

    dna = features.feature_ecommerce_metric(job, dna)

    dna = features.feature_distinct_days(job, dna)

    dna = features.feature_transactions(job, dna)

    dna = dna.drop(
        *[
            col
            for col in orig_cols
            if col not in ["MBRSHP_SID", "FISCAL_WEEK_END"]
        ]
    )

    return dna

In [0]:
job = job_manager.JobManager(spark, intermediate_all_tables_dict, member_dna_config_path)

In [0]:
recency_lookback_duration = job.config["params"].get(
    "recency_lookback_duration", {}
)
member_dna_input_data_validator(
    silver_transaction_fiscal_header, silver_transaction_fiscal_detail, silver_transaction_fiscal_detail_isnr, silver_skeleton, silver_master_member_extended,
    recency_lookback_duration=recency_lookback_duration,
    spark=spark
)


In [0]:
job.read_table("header_fiscal") # job.read
job.read_table("detail_fiscal")
job.read_table("detail_isnr_fiscal")
job.read_table("skeleton") 
job.read_table("member_extended")

In [0]:
job.tables["header_fiscal"] = gp.apply_fw_date_range(
    job, job.tables["header_fiscal"] # job.data.tables
)

job.tables["detail_fiscal"] = gp.apply_fw_date_range(
    job, job.tables["detail_fiscal"]
)

job.tables["detail_isnr_fiscal"] = gp.apply_fw_date_range(
    job, job.tables["detail_isnr_fiscal"]
)

population = gp.generate_population(job)
job.tables["population"] = population

features = generate_transaction(job)

### Save results

In [0]:
spark.sql(f"DELETE FROM {fs_cubes_transaction_1}")

fe = FeatureEngineeringClient()

fe.write_table(
    name=fs_cubes_transaction_1,
    df=features,
    mode="merge"
)